In [30]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

'''
Created on 2024-08-12
Last modified on 2024-08-12
@author: Juan Enrique López
@description: Jupyter Notebook creado para obtener reglas de detección asociadas a una lista de TTP's así como la generación de un archivo .md con la información complementaria a la solicitud.

'''

"\nCreated on 2024-08-12\nLast modified on 2024-08-12\n@author: Juan Enrique López\n@description: Jupyter Notebook creado para obtener reglas de detección asociadas a una lista de TTP's así como la generación de un archivo .md con la información complementaria a la solicitud.\n\n"

**IMPORTANTE - REQUERIMIENTOS PREVIOS**

- Requiere haber ejecutado previamente dentro de este mismo módulo **get_rules_and_classify_by_ttp** el notebook **get_rules_and_classify_by_ttp.ipynb** y disponer de la carpeta "outputs". 

- Imprescindible: ejecución del módulo **mitre_relationships** con los outputs correspondientes a stix2 (enterprise, ics, mobile...information, relations...techniques_tactics, techniques_groups, techniques_platforms, techniques_datasources, techniques_software...). Para la generación del archivo .md informativo.

In [31]:
import os
import pandas as pd
import shutil
# import requests
from stix2 import Filter, MemoryStore
# import stix2

##### Funciones

In [32]:
def list_files_in_directory(directory_path):
    """
    Lista todos los archivos en una ruta dada, con manejo de errores
    si hay más de un archivo o la carpeta está vacía. Retorna la ruta completa del archivo.

    :param directory_path: Ruta de la carpeta que deseas listar.
    :return: Ruta completa del archivo o un mensaje de error.
    """
    try:
        # Listar todos los archivos y directorios en la ruta dada
        files = os.listdir(directory_path)
        # Filtrar para obtener solo archivos (no carpetas)
        files = [f for f in files if os.path.isfile(os.path.join(directory_path, f))]
        
        if not files:
            raise ValueError(f"La carpeta en la ruta '{directory_path}' está vacía.")
        
        if len(files) > 1:
            raise ValueError(f"Se encontró más de un archivo en la carpeta '{directory_path}'.")
        
        return os.path.join(directory_path, files[0])
    
    except FileNotFoundError:
        return f"La ruta '{directory_path}' no existe."
    
    except ValueError as ve:
        return str(ve)

In [33]:
def get_rules_df(folder_source, list_ttp, path_rules):
    '''
     Función cuyo cometido es crear un dataframe que contenga el id de la ttp, nombre de la regla, origen y ruta.
    '''
    # Crear un DataFrame vacío con las columnas 'ttp', 'rule', 'source', 'path'
    df = pd.DataFrame(columns=['technique ID', 'rule', 'source', 'path'])
    # Iterar sobre las carpetas de origen
    for source in folder_source:
        # Iterar sobre la lista de TTP
        for ttp in list_ttp:
            # Construir la ruta a las reglas
            rules = os.path.join(path_rules, source, ttp)
            # Comprobar si la ruta existe
            if os.path.exists(rules):
                # Listar los archivos en la carpeta de reglas
                rule_list = os.listdir(rules)
                # Crear las rutas completas para cada regla
                full_paths = [os.path.join(rules, rule) for rule in rule_list]
                # Crear un DataFrame temporal con la información actual
                temp_df = pd.DataFrame({
                    'technique ID': [ttp] * len(rule_list),
                    'rule': rule_list,
                    'source': [source] * len(rule_list),
                    'path': full_paths
                })
                # Concatenar el DataFrame temporal con el principal
                df = pd.concat([df, temp_df], ignore_index=True)

    # Ordenar el DataFrame por el campo 'ttp' en orden descendente y resetear el índice
    return df.sort_values(by='technique ID', ascending=False).reset_index(drop=True)

In [34]:
def copy_files(df, path_output, optional_folder):
    '''
     Función encargada de rercorrer el dataframe y copiar los archivos a una nueva estructura de carpetas.
    '''
    # Recorrer cada fila del DataFrame
    for _, row in df.iterrows():
        # Obtener el ttp, rule y la ruta completa del archivo
        ttp = row['technique ID']
        rule = row['rule']
        source_path = row['path']

        # Crear la ruta destino en la estructura output/[technique ID]/[rule]
        destination_dir = os.path.join(path_output, optional_folder, ttp)
        destination_path = os.path.join(destination_dir, rule)

        # Crear las carpetas si no existen
        os.makedirs(destination_dir, exist_ok=True)

        # Copiar el archivo al destino
        shutil.copy(source_path, destination_path)
        print(f"Archivo copiado: {source_path} -> {destination_path}")

In [35]:
def get_data_from_techniques(matrix):
    '''
    Función cuyo cometido es leer el archivo csv de técnicas generado por el módulo mitre_relationships y el nb [stix2]_mitre_relationships.ipynb. El archivo consultado se corresponderá con la matriz facilitada en el argumento obligatorio.
    '''
    file_name = f'[MITRE]_{matrix}_techniques.csv'
    path_file = os.path.join(os.path.dirname(os.getcwd()), 'mitre_relationships', 'outputs\stix2', matrix, r'information\techniques',file_name)
    information_df = pd.read_csv(path_file, sep=';', quotechar='"')
    return information_df

In [36]:
def get_data_from_relation_techniques_tactics(matrix):
    '''
    Función cuyo cometido es leer el archivo csv de técnicas generado por el módulo mitre_relationships y el nb [stix2]_mitre_relationships.ipynb. El archivo consultado se corresponderá con la matriz facilitada en el argumento obligatorio.
    '''
    file_name = f'[MITRE]_{matrix}_technique_tactics_1N.csv'
    path_file = os.path.join(os.path.dirname(os.getcwd()), 'mitre_relationships', 'outputs\stix2', matrix, r'relations\techniques_tactics',file_name)
    information_df = pd.read_csv(path_file, sep=';', quotechar='"')
    return information_df

In [37]:
def get_data_from_relation_techniques_software(matrix):
    '''
    Función cuyo cometido es leer el archivo csv de técnicas generado por el módulo mitre_relationships y el nb [stix2]_mitre_relationships.ipynb. El archivo consultado se corresponderá con la matriz facilitada en el argumento obligatorio.
    '''
    file_name = f'[MITRE]_{matrix}_technique_software_1N.csv'
    path_file = os.path.join(os.path.dirname(os.getcwd()), 'mitre_relationships', 'outputs\stix2', matrix, r'relations\techniques_software',file_name)
    information_df = pd.read_csv(path_file, sep=';', quotechar='"')
    return information_df

In [38]:
def get_data_from_relation_techniques_platforms(matrix):
    '''
    Función cuyo cometido es leer el archivo csv de técnicas generado por el módulo mitre_relationships y el nb [stix2]_mitre_relationships.ipynb. El archivo consultado se corresponderá con la matriz facilitada en el argumento obligatorio.
    '''
    file_name = f'[MITRE]_{matrix}_technique_platforms_1N.csv'
    path_file = os.path.join(os.path.dirname(os.getcwd()), 'mitre_relationships', 'outputs\stix2', matrix, r'relations\techniques_platforms',file_name)
    information_df = pd.read_csv(path_file, sep=';', quotechar='"')
    return information_df


In [39]:
def get_data_from_relation_techniques_groups(matrix):
    '''
    Función cuyo cometido es leer el archivo csv de técnicas generado por el módulo mitre_relationships y el nb [stix2]_mitre_relationships.ipynb. El archivo consultado se corresponderá con la matriz facilitada en el argumento obligatorio.
    '''
    file_name = f'[MITRE]_{matrix}_technique_groups_1N.csv'
    path_file = os.path.join(os.path.dirname(os.getcwd()), 'mitre_relationships', 'outputs\stix2', matrix, r'relations\techniques_groups',file_name)
    information_df = pd.read_csv(path_file, sep=';', quotechar='"')
    return information_df

In [40]:
def get_data_from_relation_techniques_datasources(matrix):
    '''
    Función cuyo cometido es leer el archivo csv de técnicas generado por el módulo mitre_relationships y el nb [stix2]_mitre_relationships.ipynb. El archivo consultado se corresponderá con la matriz facilitada en el argumento obligatorio.
    '''
    file_name = f'[MITRE]_{matrix}_technique_datasources_1N.csv'
    path_file = os.path.join(os.path.dirname(os.getcwd()), 'mitre_relationships', 'outputs\stix2', matrix, r'relations\techniques_datasources',file_name)
    information_df = pd.read_csv(path_file, sep=';', quotechar='"')
    return information_df

In [41]:
def format_cols_for_md(column):
    '''
    Función para aplicar los reemplazos que permitan en el .md relacionar elementos.
    '''
    if column.name == 'technique_ID':
        column = column.apply(lambda x: f'[[{x}]]')
    elif column.dtype == "object":  # Solo aplica a columnas de tipo string
        column = column.str.replace("'", '', regex=False)
        column = column.str.replace("[", '[[', regex=False)
        column = column.str.replace("]", ']]', regex=False)
        column = column.str.replace(", ", ']] [[', regex=False)
    return column

In [42]:
def replace_chars(column):
    if column.dtype == "object":  # Verifica que la columna sea de tipo string
        column = column.str.replace("['", '', regex=False)
        column = column.str.replace("']", '', regex=False)
        column = column.str.replace("'", '', regex=False)
    return column

In [43]:
def generate_markdown(df_techniques, resume_rules_df, df_rules, folder_path, optional_folder=''):
    # Asegúrate de que la carpeta exista
    os.makedirs(folder_path, exist_ok=True)
    
    # Construir la ruta completa del archivo
    filename = f'riesgos_{optional_folder}.md'  # Nombre fijo para el archivo Markdown
    file_path = os.path.join(folder_path, filename)
    
    with open(file_path, 'w') as file:
        file.write("### Resumen reglas de detección disponibles:\n\n")

        file.write("| technique ID | source | rule |\n")
        file.write("| ------------ | ------ | ---- |\n")
        for _, row in resume_rules_df.iterrows():
            file.write(f"| {row['technique ID']} | {row['source']} | {row['rules']} |\n")
        file.write("\n\n")

        file.write("### Desglose reglas de detección disponibles:\n\n")

        file.write("| technique ID | source | rules |\n")
        file.write("| ------------ | ------ | ---- |\n")
        for _, row in df_rules.iterrows():
            file.write(f"| {row['technique ID']} | {row['source']} | {row['rule']} |\n")
        file.write("\n\n")
        
        # Agregar el resto del contenido del DataFrame df_techniques
        for index, row in df_techniques.iterrows():
            # Título de la técnica
            file.write(f"### {row['technique ID'].strip('[]')}\n\n")
            
            # Información de la técnica
            file.write(f"**technique ID**\n{row['technique ID']}\n\n")
            file.write(f"**technique**\n{row['technique']}\n\n")
            file.write(f"**technique url**\n{row['technique url']}\n\n")
            file.write(f"**technique description**\n{row['technique description']}\n\n")
            file.write(f"**technique deprecated**\n{row['technique deprecated']}\n\n")
            file.write(f"**technique revoked**\n{row['technique revoked']}\n\n")
            file.write(f"**matrix domains**\n{row['matrix domains']}\n\n")
            file.write(f"**tactic ID**\n{row['tactic ID']}\n\n")
            file.write(f"**tactic**\n{row['tactic']}\n\n")
            file.write(f"**software ID**\n{row['software ID']}\n\n")
            file.write(f"**software**\n{row['software']}\n\n")
            file.write(f"**platform**\n{row['platform']}\n\n")
            file.write(f"**group ID**\n{row['group ID']}\n\n")
            file.write(f"**group**\n{row['group']}\n\n")
            file.write(f"**data source ID**\n{row['data source ID']}\n\n")
            file.write(f"**data source**\n{row['data source']}\n\n")

            # Añadir una línea en blanco para separar secciones
            file.write("\n\n")

    print(f"Archivo '{filename}' creado en la carpeta '{folder_path}' con éxito.")


In [44]:
# def info_techniques(results_df, mitre_domain):
#     techniques_df = get_data_from_techniques(mitre_domain)
#     techniques_df = techniques_df[techniques_df['technique_ID'].isin(results_df['technique ID'].unique().tolist())]
#     techniques_df = techniques_df.sort_values(by='technique_ID', ascending=False).reset_index(drop=True)
#     column_list = ['technique_ID','tactic_ID','software_ID','platform','group_ID','data_source_ID']
#     for col in techniques_df.columns:
#         if col in column_list:
#             techniques_df[col] = techniques_df[col].apply(lambda x: f'[[{x}]]')
#         else:
#             techniques_df[col] = replace_chars(techniques_df[col])            
#     return techniques_df

def info_techniques(results_df, mitre_domain):
    techniques_df = get_data_from_techniques(mitre_domain)
    techniques_df = techniques_df[techniques_df['technique_ID'].isin(results_df['technique ID'].unique().tolist())]
    techniques_df = techniques_df.sort_values(by='technique_ID', ascending=False).reset_index(drop=True)
    
    column_list = ['technique_ID', 'tactic_ID', 'software_ID', 'platform', 'group_ID', 'data_source_ID']
    for col in techniques_df.columns:
        if col in column_list:
            techniques_df[col] = techniques_df[col].apply(lambda x: f'[[{x}]]')
        elif techniques_df[col].dtype == "object":  # Solo aplicar si es string
            techniques_df[col] = replace_chars(techniques_df[col])
    
    return techniques_df

In [45]:
def info_techniques_tactics(results_df, mitre_domain):
    techniques_tactics_df = get_data_from_relation_techniques_tactics(mitre_domain)
    techniques_tactics_df = techniques_tactics_df[techniques_tactics_df['technique_ID'].isin(results_df['technique ID'].unique().tolist())]
    techniques_tactics_df = techniques_tactics_df.sort_values(by='technique_ID', ascending=False).reset_index(drop=True)
    column_list = ['technique_ID','tactic_ID','software_ID','platform','group_ID','data_source_ID']
    for col in techniques_tactics_df.columns:
        if col in column_list:
            techniques_tactics_df[col] = format_cols_for_md(techniques_tactics_df[col])
        else:
           techniques_tactics_df[col] =  replace_chars(techniques_tactics_df[col])
    return techniques_tactics_df

In [46]:
def info_techniques_software(results_df, mitre_domain):
    techniques_software_df = get_data_from_relation_techniques_software(mitre_domain)
    techniques_software_df = techniques_software_df[techniques_software_df['technique_ID'].isin(results_df['technique ID'].unique().tolist())]
    techniques_software_df = techniques_software_df.sort_values(by='technique_ID', ascending=False).reset_index(drop=True)
    column_list = ['technique_ID','tactic_ID','software_ID','platform','group_ID','data_source_ID']
    for col in techniques_software_df.columns:
        if col in column_list:
            techniques_software_df[col] = format_cols_for_md(techniques_software_df[col])
        else:
           techniques_software_df[col] =  replace_chars(techniques_software_df[col])
    return techniques_software_df

In [47]:
def info_techniques_platforms(results_df, mitre_domain):
    techniques_platforms_df = get_data_from_relation_techniques_platforms(mitre_domain)
    techniques_platforms_df = techniques_platforms_df[techniques_platforms_df['technique_ID'].isin(results_df['technique ID'].unique().tolist())]
    techniques_platforms_df = techniques_platforms_df.sort_values(by='technique_ID', ascending=False).reset_index(drop=True)
    column_list = ['technique_ID','tactic_ID','platform_ID','platform','group_ID','data_source_ID']
    for col in techniques_platforms_df.columns:
        if col in column_list:
            techniques_platforms_df[col] = format_cols_for_md(techniques_platforms_df[col])
        else:
           techniques_platforms_df[col] =  replace_chars(techniques_platforms_df[col])
    return techniques_platforms_df

In [48]:
def info_techniques_groups(results_df, mitre_domain):
    techniques_groups_df = get_data_from_relation_techniques_groups(mitre_domain)
    techniques_groups_df = techniques_groups_df[techniques_groups_df['technique_ID'].isin(results_df['technique ID'].unique().tolist())]
    techniques_groups_df = techniques_groups_df.sort_values(by='technique_ID', ascending=False).reset_index(drop=True)
    column_list = ['technique_ID','tactic_ID','platform_ID','platform','group_ID','data_source_ID']
    for col in techniques_groups_df.columns:
        if col in column_list:
            techniques_groups_df[col] = format_cols_for_md(techniques_groups_df[col])
        else:
           techniques_groups_df[col] =  replace_chars(techniques_groups_df[col])
    return techniques_groups_df

In [49]:
def info_techniques_datasources(results_df, mitre_domain):
    techniques_datasources_df = get_data_from_relation_techniques_datasources(mitre_domain)
    techniques_datasources_df = techniques_datasources_df[techniques_datasources_df['technique_ID'].isin(results_df['technique ID'].unique().tolist())]
    techniques_datasources_df = techniques_datasources_df.sort_values(by='technique_ID', ascending=False).reset_index(drop=True)
    column_list = ['technique_ID','tactic_ID','platform_ID','platform','group_ID','data_source_ID']
    for col in techniques_datasources_df.columns:
        if col in column_list:
            techniques_datasources_df[col] = format_cols_for_md(techniques_datasources_df[col])
        else:
           techniques_datasources_df[col] =  replace_chars(techniques_datasources_df[col])
    return techniques_datasources_df

**Parámetros iniciales**

In [50]:
input_file = list_files_in_directory(os.path.join(os.getcwd(),'input_ttps'))
mitre_domain = 'enterprise' # enterprise, mobile, ics

In [51]:
# Parámetros NO modificables
rules_path = os.path.join(os.getcwd(), 'outputs', mitre_domain)
output_path = os.path.join(os.getcwd(), 'query_outputs')
source_folders = [folder for folder in os.listdir(rules_path) if os.path.isdir(os.path.join(rules_path, folder))]

In [52]:
input_df = pd.read_excel(input_file)
input_df.head(4)

,RIESGO,TTP
0,SMS-Unknown-data-loc,T1083
1,SMS-Unknown-data-loc,T1056
2,SMS-Unknown-data-loc,T1041
3,DEV/OPS-Customer-Separation,T1190


In [53]:
input_df = input_df.groupby('RIESGO').agg({'TTP': lambda x: list(x)}).reset_index()
input_df.head(3)

,RIESGO,TTP
0,Compromise-EUP,"[T1078, T1059, T1485]"
1,Compromise-Wide,"[T1078, T1059, T1485]"
2,DEV/OPS-Customer-Separation,"[T1190, T1075, T1485]"


#### **Ejecución**

In [54]:
for index, row in input_df.iterrows():
    # 1. Bucle para recorrer el dataframe de riesgos - ttps
    optional_folder = row['RIESGO'].replace("\\", "-").replace("/", "-")
    ttp_list = row['TTP']
    results_df = get_rules_df(source_folders, ttp_list, rules_path)

    # 2. Generación de la estructura de carpetas por ttp con las reglas de detección
    copy_files(results_df, output_path, optional_folder)

    # 3. Obtención de la información relativa a la consulta
    techniques_df = info_techniques(results_df, mitre_domain)
    techniques_tactics_df = info_techniques_tactics(results_df, mitre_domain)
    techniques_software_df = info_techniques_software(results_df, mitre_domain)
    techniques_platforms_df = info_techniques_platforms(results_df, mitre_domain)
    techniques_groups_df = info_techniques_groups(results_df, mitre_domain)
    techniques_datasources_df = info_techniques_datasources(results_df, mitre_domain)

    # 4. Unión de las tablas informativas y formateos previos a la generación del .md
    info_df = pd.merge(techniques_tactics_df, techniques_software_df, on=['technique_ID', 'technique'], how='left')
    info_df = pd.merge(info_df, techniques_platforms_df, on=['technique_ID', 'technique'], how='left')
    info_df = pd.merge(info_df, techniques_groups_df, on=['technique_ID', 'technique'], how='left')
    info_df = pd.merge(info_df, techniques_datasources_df, on=['technique_ID', 'technique'], how='left')
    info_df = pd.merge(info_df, techniques_df, on=['technique_ID', 'technique'], how='left')
    info_df.columns = info_df.columns.str.replace('_', ' ', regex=False)

    # 5. Formateo de la tabla resumen de reglas de detección por técnica
    results_df = results_df[['technique ID', 'source', 'rule']]

    # 6. Generación de la tabla resumen inicial
    resume_results_df = results_df.groupby(['technique ID', 'source'])['rule'].count().reset_index(name='rules')

    # 7. Guardado del archivo markdown con la información relativa a la query
    generate_markdown(info_df, resume_results_df, results_df, os.path.join(output_path, optional_folder), optional_folder)

    print('Proceso finalizado correctamente!')


Archivo copiado: c:\Users\jelopez\Documents\CyberProof\python\develop\get_rules_and_classify_by_ttp\outputs\enterprise\UCM Catalog 2024 Splunk\T1485\T1485 - CyberProof - All - Logs Not Being Indexed.md -> c:\Users\jelopez\Documents\CyberProof\python\develop\get_rules_and_classify_by_ttp\query_outputs\Compromise-EUP\T1485\T1485 - CyberProof - All - Logs Not Being Indexed.md
Archivo copiado: c:\Users\jelopez\Documents\CyberProof\python\develop\get_rules_and_classify_by_ttp\outputs\enterprise\Mappings\T1485\AWSCloudEndure.yaml -> c:\Users\jelopez\Documents\CyberProof\python\develop\get_rules_and_classify_by_ttp\query_outputs\Compromise-EUP\T1485\AWSCloudEndure.yaml
Archivo copiado: c:\Users\jelopez\Documents\CyberProof\python\develop\get_rules_and_classify_by_ttp\outputs\enterprise\Elastic 1\T1485\impact_ec2_disable_ebs_encryption.toml -> c:\Users\jelopez\Documents\CyberProof\python\develop\get_rules_and_classify_by_ttp\query_outputs\Compromise-EUP\T1485\impact_ec2_disable_ebs_encryption.